In [31]:
**Importing the modules**
import json,os,time 
from dotenv import load_dotenv 
from google.genai import Client
from pageindex import PageIndexClient


load_dotenv()

SyntaxError: invalid syntax (456563248.py, line 1)

**Loading Configs**

In [ ]:
PAGEINDEX_API_KEY=os.getenv("PAGEINDEX_API_KEY")
GEMINI_API_KEY=os.getenv("GEMINI_API_KEY")


pg_index_client=PageIndexClient(api_key=PAGEINDEX_API_KEY) #type: ignore 
gemini_client=Client(api_key=GEMINI_API_KEY) # type: ignore 

**Pdf Upload to page index cloud**

In [ ]:
file_path=r"C:\Users\Devab\Downloads\acm_club_app_fresh_plan.pdf"

print("Uploading file ...")
res=pg_index_client.submit_document(file_path=file_path)
doc_id=res["doc_id"]

print("Uploaded Doc")

Uploading file ...
Uploaded Doc


**Checking the index  creation**

In [ ]:
print("Building Index...")

while True:
    status_res=pg_index_client.get_document(doc_id)
    status=status_res.get("status")
    
    print(f"Status: {status}")
    
    if status=="completed":
        print("Tree Index is built")
        break
    elif status=="failed":
        print("Tree Index failed")
        break
    
    time.sleep(2)

Building Index...
Status: processing
Status: processing
Status: processing
Status: processing
Status: processing
Status: completed
Tree Index is built


**Printing  the tree structure**

In [ ]:
tree_result  = pg_index_client.get_tree(doc_id, node_summary=True)
pageindex_tree = tree_result.get("result", [])

def print_tree(nodes,indent=0):
    for node in nodes:
        prefix=" "*indent + ("└─ " if indent > 0 else "")
        page=node.get("page_index","?")
        print(f"{prefix}[{node['node_id']}] {node['title']} (p.{page})")
        
        if node.get("nodes"):
            print_tree(node["nodes"],indent+1)

print("Document Structure")
print_tree(pageindex_tree)

Document Structure
[0000] ACM Club App Project Plan (p.1)
 └─ [0001] Overview (p.1)
 └─ [0002] Main Goal (p.1)
 └─ [0003] What the App Will Do (p.1)
 └─ [0004] User Roles (p.1)
 └─ [0005] Member Experience (p.2)
 └─ [0006] Admin Experience (p.2)
 └─ [0007] Core Modules (p.2)
 └─ [0008] Member Management (p.2)
 └─ [0009] Attendance (p.2)
 └─ [0010] Leave Management (p.2)
 └─ [0011] Announcements and Updates (p.2)
 └─ [0012] Resources (p.2)
 └─ [0013] Leaderboards (p.3)
 └─ [0014] Notifications (p.3)
 └─ [0015] Project Structure (p.3)
 └─ [0016] Rollout Plan (p.3)
 └─ [0017] Design Direction (p.3)
 └─ [0018] Future Expansion (p.3)


In [ ]:
def count_nodes(nodes):
    total = len(nodes)
    for n in nodes:
        if n.get("nodes"):
            total += count_nodes(n["nodes"])
    return total

total = count_nodes(pageindex_tree)
print(f"Total nodes in tree: {total}")

Total nodes in tree: 19


**E2E RAG Pipeline**

In [ ]:
def llm_tree_search(query: str, tree: list, model: str = "gemini-3.1-flash-lite-preview") -> dict:

    def compress(nodes):
        out = []
        for n in nodes:
            entry = {
                "node_id": n["node_id"],
                "title":   n["title"],
                "page":    n.get("page_index", "?"),
                "summary": n.get("text", "")[:150]  # first 150 chars
            }
            if n.get("nodes"):
                entry["children"] = compress(n["nodes"])
            out.append(entry)
        return out
    
    compressed_tree = compress(tree)
    
    prompt = f"""You are given a query and a document's tree structure (like a Table of Contents).
Your task: identify which node IDs most likely contain the answer to the query.
Think step-by-step about which sections are relevant.

Query: {query}

Document Tree:
{json.dumps(compressed_tree, indent=2)}

Reply ONLY in this exact JSON format:
{{
  "thinking": "<your step-by-step reasoning>",
  "node_list": ["node_id1", "node_id2"]
}}"""

    response = gemini_client.models.generate_content(
        model=model,
        contents=prompt,
        config={
            "temperature": 0.2,
            "response_mime_type": "application/json",
        },
    )
    
    return json.loads(response.text) #type: ignore 

In [ ]:
def find_nodes(tree,target_ids):
    found=[]
    for node in tree:
        if node["node_id"] in target_ids:
            found.append(node)
        if node.get("nodes"):
            found.extend(find_nodes(node["nodes"],target_ids=target_ids))
    
    return found  


def generate_answer(query,nodes):
    if not nodes:
        return "No relevant sections found in the doc"
    
    context_parts=[]
    for node in nodes:
        context_parts.append(f"[Section: '{node['title']}' | Page {node.get('page_index', '?')}]\n"
            f"{node.get('text', 'Content not available.')}")
        
    context="\n\n---\n\n".join(context_parts)
    
    prompt=f"""You are an expert document analyst.Answer the question using ONLY the provided context.For every claim you make, cite the section title and page number in parentheses.Be concise and precise.

    Question: {query}
    Context:
    {context}
    Answer:"""
    
    response=gemini_client.models.generate_content(
        model='gemini-3.1-flash-lite-preview',
        contents=prompt,
        config={
            'temperature':0.2,
            'top_k':40,
            'top_p':0.95,
        },
    )
    
    return response.text  #type: ignore

**Main Rag Function**

In [ ]:
def vectorless_rag(query: str, tree: list, verbose: bool = True) -> str:
    """
    Step 1: LLM Tree Search  → finds relevant node_ids
    Step 2: Node Retrieval   → fetches section content
    Step 3: Answer Generation → produces cited answer
    """
    if verbose:
        print(f"{'='*55}")
        print(f" Query: {query}")
        print(f"{'='*55}")
    
    # Step 1: Tree Search
    search_result  = llm_tree_search(query, tree)
    node_ids       = search_result.get("node_list", [])
    
    if verbose:
        print(f"\n Reasoning: {search_result.get('thinking', '')[:200]}...")
        print(f" Retrieved node IDs: {node_ids}")
    
    # Step 2: Retrieve nodes
    nodes = find_nodes(tree, node_ids)
    
    if verbose:
        print(f"Sections found: {[n['title'] for n in nodes]}")
    
    # Step 3: Generate answer
    answer = generate_answer(query, nodes)
    
    if verbose:
        print(f"\nAnswer:\n{answer}")
    
    return answer

**Testing**

In [ ]:
# ── Test with multiple queries ───────────────────────────────────────────────
test_queries = [
    "What is the main goal of the ACM Club App project plan?",
    "What will the app do for members?",
    "What does the admin experience include?",
    "How will attendance be handled in the app?",
    "What is included in the rollout plan?",
]

for q in test_queries:
    print()
    ans = vectorless_rag(q, pageindex_tree, verbose=False)
    print(f"Q: {q}")
    print(f"A: {ans[:300]}...")
    print("-" * 55)


Q: What is the main goal of the ACM Club App project plan?
A: The main goal of the ACM Club App project is to replace scattered communication and manual tracking with a single, organized system for managing the club (Main Goal, Page 1)....
-------------------------------------------------------

Q: What will the app do for members?
A: For members, the app serves as the primary interface to perform the following actions:

*   **Registration and Profile Management:** Members can register and manage their profiles ("What the App Will Do," Page 1).
*   **Attendance:** Members can track attendance by scanning daily QR codes ("What the...
-------------------------------------------------------

Q: What does the admin experience include?
A: The admin experience includes a web dashboard used by coordinators and SIG leads to manage club operations, specifically supporting the following tasks:

*   Approving members (Admin Experience, Page 2)
*   Posting announcements (Admin Experience, Page 2)